# AI-WAF Training and Safe Evaluation (Google Colab / Local)

This notebook runs the mandatory Logistic Regression pipeline only. It verifies canonical training readiness, fits character TF-IDF and Logistic Regression, calibrates on validation data, and saves checksum-verified artifacts. Final evaluation stays locked unless the canonical manifest explicitly authorizes it.


## 1. Environment Setup

In [ ]:
# Dependencies must be installed explicitly in the selected environment before this cell.
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn

print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")


## 2. Verify Canonical Training Readiness

Raw-only splits are not training-ready. The current raw preparation path records `training_ready: false`; do not edit manifest flags. This check reads only canonical train and validation files and verifies their manifest, checksums, labels, and leakage constraints.


In [ ]:
from training.train_lr import load_splits

train_df, val_df = load_splits("data/splits")
print(f"Verified canonical rows: train={len(train_df)}, validation={len(val_df)}")
print("The held-out test partition was not loaded.")


## 3. Train, Calibrate, and Serialize Logistic Regression

`training.train_lr` fits character TF-IDF on training payloads, trains Logistic Regression, calibrates the fitted classifier using the validation split, and writes `vectorizer.pkl`, `model.pkl`, `calibrator.pkl`, and `checksums.sha256`. Existing artifact directories are not overwritten.


In [ ]:
import subprocess

subprocess.run(
    [
        sys.executable, "-m", "training.train_lr",
        "--splits-dir", "data/splits",
        "--output-dir", "trained_models",
    ],
    check=True,
)


## 4. Safe Evaluation (Explicit Authorization Required)

The evaluator verifies model checksums and requires `final_evaluation_authorized: true` in the canonical manifest before reading the held-out split. It uses the frozen `ai.threshold_block` value from `config.yaml`, reports classification and calibration metrics, and saves figures. It does not tune the threshold or generate a Markdown report.


In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "training.evaluate",
        "--splits-dir", "data/splits",
        "--models-dir", "trained_models",
        "--output-dir", "evaluation_artifacts",
    ],
    check=True,
)
